In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,RobustScaler
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping

df = pd.read_csv('training_data_ready.csv')
df = df.drop([21,43,92,159], axis=0) #[21,43,92,159] # [92,99,31,169,150,43] #[17,21,43,53,62,92,129,150,159,166,169] #[15,17,21,30,35,43,53,62,67,72,73,75,91,92,129,139,142,150,159,166,169]
target_cols = ['x', 'y', 'z'] 
x_cols = [col for col in df.columns if col not in target_cols + ['source_file','Objektname','Objektkategorie','OAufnahme','Schwierigkeit','Gewicht']]
    
X = df[x_cols]
y = df[target_cols]
    # 3. Train-Test-Split (80% Training, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [25]:
#---------------------------- Ausgaben Beste Modelle: -----------------------
#Platz  | Scaler    | Netz-Struktur      | L2     | Dropout | LR     | Val-Loss (MSE) | Val-MAE 
#-----------------------------------------------------------------------------------------------
#1    | robust    | [128, 64]          | 0.1    | 0.0     | 0.05   | 314.078888     | 12.3596 
#2    | robust    | [512, 256, 128, 64] | 0.5    | 0.0     | 0.05   | 322.648865     | 12.1665 
#3    | robust    | [512, 256, 128, 64] | 0.1    | 0.0     | 0.01   | 348.768555     | 12.1428 
#4    | robust    | [256, 128, 64]     | 0.1    | 0.0     | 0.01   | 351.149323     | 11.6752 
#5    | minmax    | [512, 256, 128, 64] | 0.0    | 0.1     | 0.005  | 368.512878     | 12.4263 




def create_neural_network(X_train, y_train):

    # 5. Architektur des Neuronalen Netzes definieren
    model = models.Sequential([
        # Eingabeschicht (Größe entspricht Anzahl der Features, z.B. 28)
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dropout(0.1),

        layers.Dense(512, activation='relu'),#, kernel_regularizer=regularizers.l2(0.5)),
         # 1 versteckte Schicht
        layers.Dense(256, activation='relu'),#, kernel_regularizer=regularizers.l2(0.5)),
        layers.Dropout(0.1), # Verhindert Overfitting

        # 2 versteckte Schicht
        layers.Dense(128, activation='relu'),#, kernel_regularizer=regularizers.l2(0.5)),
        layers.Dropout(0.1), # Verhindert Overfitting
        
        # 3 versteckte Schicht
        layers.Dense(64, activation='relu'),#, kernel_regularizer=regularizers.l2(0.5)),
        layers.Dropout(0.1), # Verhindert Overfitting
        
        # Ausgabeschicht: 3 Neuronen für x, y, z (keine Aktivierungsfunktion für Regression)
        layers.Dense(3)
    ])
    
    # 6. Kompilieren
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.005)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    
    # 7. Training
    print("Starte Training...")

    # 3. Early Stopping
    early_stopping = EarlyStopping(
        monitor='val_loss', 
        patience=500, 
        restore_best_weights=True,
        verbose=0
    )
    history = model.fit(
        X_train_scaled, y_train,
        epochs=5000,
        batch_size=8,
        validation_split=0.1,
        callbacks=[early_stopping],
        verbose=0
    )

        # 5. Besten Validierungsfehler auslesen
    min_val_loss = min(history.history['val_loss'])
    corresponding_mae = history.history['val_mae'][history.history['val_loss'].index(min_val_loss)]
    epochs_trained = len(history.history['val_loss'])
    
    print(f" -> Fertig nach {epochs_trained} Epochen. Bester Val-Loss: {min_val_loss:.4f} mit MAE: {corresponding_mae:.4f}")
    
    return model

In [26]:
model = create_neural_network(X_train_scaled, y_train)

Starte Training...
 -> Fertig nach 585 Epochen. Bester Val-Loss: 131.7185 mit MAE: 8.1528


In [27]:
import datetime
import re
import numpy as np
import pandas as pd
import tensorflow as tf  # Wichtig, um die Learning Rate sauber auszulesen

# --- Dein bestehender Code für die Vorhersage ---
y_pred = model.predict(X_test_scaled)

y_pred_df = pd.DataFrame(
    y_pred, columns=["x_pred", "y_pred", "z_pred"], index=y_test.index
)

results = y_test.copy()
results[["x_pred", "y_pred", "z_pred"]] = y_pred_df

# Fehler berechnen
results["abs_error_x"] = np.abs(results["x_pred"] - results["x"])
results["abs_error_y"] = np.abs(results["y_pred"] - results["y"])
results["abs_error_z"] = np.abs(results["z_pred"] - results["z"])

results["mse_x"] = (results["x_pred"] - results["x"]) ** 2
results["mse_y"] = (results["y_pred"] - results["y"]) ** 2
results["mse_z"] = (results["z_pred"] - results["z"]) ** 2

# --- Restliche Spalten aus 'df' anhängen ---
#restliche_spalten = df.loc[y_test.index].drop(
#    columns=["x", "y", "z"], errors="ignore"
#)
#doppelte_spalten = [c for c in restliche_spalten.columns if c in results.columns]
#restliche_spalten = restliche_spalten.drop(columns=doppelte_spalten)

results = pd.concat([results, df.loc[y_test.index, ["source_file"]]], axis=1)


# --- NEU: Keras-Parameter dynamisch extrahieren ---

# 1. Zeitstempel erzeugen (Format: JYYMMDD_HHMMSS)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# 2. Modelltyp (wird "Sequential" sein)
model_name = model.__class__.__name__

try:
    # Netzarchitektur auslesen: Holt die Neuronenanzahl aller Dense-Schichten
    # Erzeugt eine Liste wie ['128', '64', '3']
    dense_units = [
        str(layer.units) for layer in model.layers if hasattr(layer, "units")
    ]
    arch_str = "-".join(dense_units)  # Wird zu "128-64-3"

    # Optimizer-Name auslesen (z.B. "Adam")
    opt_name = model.optimizer.__class__.__name__

    # Learning Rate auslesen (Keras speichert diese intern oft als Variable/Tensor)
    try:
        lr = float(tf.keras.backend.get_value(model.optimizer.learning_rate))
    except Exception:
        try:
            lr = float(model.optimizer.learning_rate.numpy())
        except Exception:
            lr = "unknown"

    # String für die Parameter zusammensetzen
    param_str = f"Arch_{arch_str}_{opt_name}_lr_{lr}"

except Exception:
    # Sicherer Fallback, falls die Keras-Version eine andere Struktur erzwingt
    param_str = "keras_nn"

# Dateiname zusammenbauen und ungültige Zeichen entfernen
filename = f"Auswertung_test_{model_name}_{param_str}_{timestamp}.csv"
filename = re.sub(r'[\\/*?:"<>| ]', "_", filename)

# Als CSV speichern (inklusive Index)
#results.to_csv(filename, index=True, sep=";", decimal=",")


# --- Konsolen-Ausgabe (unverändert bzw. erweitert) ---
print("Auswertung Testdatensatz:")
print(f"  - MAE x: {results['abs_error_x'].mean():.4f}")
print(f"  - MAE y: {results['abs_error_y'].mean():.4f}")
print(f"  - MAE z: {results['abs_error_z'].mean():.4f}")
print(f"  - MSE x: {results['mse_x'].mean():.4f}")
print(f"  - MSE y: {results['mse_y'].mean():.4f}")
print(f"  - MSE z: {results['mse_z'].mean():.4f}")

print(f"\n[INFO] Datei erfolgreich gespeichert unter: {filename}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Auswertung Testdatensatz:
  - MAE x: 16.0685
  - MAE y: 18.1626
  - MAE z: 6.8062
  - MSE x: 482.5324
  - MSE y: 677.9243
  - MSE z: 97.1057

[INFO] Datei erfolgreich gespeichert unter: Auswertung_test_Sequential_Arch_512-256-128-64-3_Adam_lr_0.004999999888241291_20260617_101751.csv


In [28]:
# Zeigt die ersten 20 Zeilen elegant an
results.style.format(
    {
        "x": "{:.2f}",
        "x_pred": "{:.2f}",
        "abs_error_x": "{:.2f}",
        "y": "{:.2f}",
        "y_pred": "{:.2f}",
        "abs_error_y": "{:.2f}",
        "z": "{:.2f}",
        "z_pred": "{:.2f}",
        "abs_error_z": "{:.2f}",
    },
    decimal=",",  # Zeigt auch im Notebook Kommas statt Punkte!
).background_gradient(
    subset=["abs_error_x", "abs_error_y", "abs_error_z"], cmap="Reds"
)

,x,y,z,x_pred,y_pred,z_pred,abs_error_x,abs_error_y,abs_error_z,mse_x,mse_y,mse_z,source_file
103,"20,00","87,50","47,50","36,13","53,15","43,52","16,13","34,35","3,98","260,285117","1179,639901","15,856737",Lumi_V_L_L_6.csv
139,"81,39","46,35","20,00","60,44","64,24","24,44","20,95","17,89","4,44","438,902398","320,085874","19,674093",Endboss_L_U_S_X_3.csv
80,"47,50","87,50","20,00","65,51","46,45","14,25","18,01","41,05","5,75","324,470935","1685,193263","33,102191",Lumi_L_L_8.csv
58,"25,00","50,00","31,75","36,87","33,38","42,23","11,87","16,62","10,48","141,012454","276,178972","109,759161",Rechteck_V_L_L_2.csv
100,"20,00","87,50","47,50","67,78","38,77","18,80","47,78","48,73","28,70","2283,183463","2375,001754","823,833909",Lumi_V_L_L_3.csv
30,"53,33","91,00","30,00","94,39","66,41","32,23","41,06","24,59","2,23","1685,714923","604,539604","4,986927",3eck_L_L_4.csv
107,"47,50","32,50","17,25","33,95","31,00","8,40","13,55","1,50","8,85","183,654069","2,254288","78,384199",Durch_L_U_D_O_4.csv
84,"47,50","87,50","20,00","68,44","65,86","17,87","20,94","21,64","2,13","438,528117","468,252922","4,548621",Lumi_L_L_12.csv
166,"12,50","120,00","17,50","70,71","21,62","14,84","58,21","98,38","2,66","3388,920065","9678,480499","7,069462",Rechteck_Lang_L_L_3.csv
111,"47,50","32,50","5,75","42,39","39,36","7,67","5,11","6,86","1,92","26,143266","47,090754","3,683854",Durch_L_U_D_U_2.csv


In [29]:
import datetime
import re
import numpy as np
import pandas as pd
import tensorflow as tf  # Wichtig, um die Learning Rate sauber auszulesen

# --- Dein bestehender Code für die Vorhersage ---
y_pred = model.predict(X_train_scaled)

y_pred_df = pd.DataFrame(
    y_pred, columns=["x_pred", "y_pred", "z_pred"], index=y_train.index
)

results = y_train.copy()
results[["x_pred", "y_pred", "z_pred"]] = y_pred_df

# Fehler berechnen
results["abs_error_x"] = np.abs(results["x_pred"] - results["x"])
results["abs_error_y"] = np.abs(results["y_pred"] - results["y"])
results["abs_error_z"] = np.abs(results["z_pred"] - results["z"])

results["mse_x"] = (results["x_pred"] - results["x"]) ** 2
results["mse_y"] = (results["y_pred"] - results["y"]) ** 2
results["mse_z"] = (results["z_pred"] - results["z"]) ** 2

# --- Restliche Spalten aus 'df' anhängen ---
##restliche_spalten = df.loc[y_train.index].drop(
#    columns=["x", "y", "z"], errors="ignore"
#)
#doppelte_spalten = [c for c in restliche_spalten.columns if c in results.columns]
#restliche_spalten = restliche_spalten.drop(columns=doppelte_spalten)

#results = pd.concat([results, restliche_spalten], axis=1)
results = pd.concat([results, df.loc[y_train.index, ["source_file"]]], axis=1)

# --- NEU: Keras-Parameter dynamisch extrahieren ---

# 1. Zeitstempel erzeugen (Format: JYYMMDD_HHMMSS)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# 2. Modelltyp (wird "Sequential" sein)
model_name = model.__class__.__name__


try:
    # Netzarchitektur auslesen: Holt die Neuronenanzahl aller Dense-Schichten
    # Erzeugt eine Liste wie ['128', '64', '3']
    dense_units = [
        str(layer.units) for layer in model.layers if hasattr(layer, "units")
    ]
    arch_str = "-".join(dense_units)  # Wird zu "128-64-3"

    # Optimizer-Name auslesen (z.B. "Adam")
    opt_name = model.optimizer.__class__.__name__

    # Learning Rate auslesen (Keras speichert diese intern oft als Variable/Tensor)
    try:
        lr = float(tf.keras.backend.get_value(model.optimizer.learning_rate))
    except Exception:
        try:
            lr = float(model.optimizer.learning_rate.numpy())
        except Exception:
            lr = "unknown"

    # String für die Parameter zusammensetzen
    param_str = f"Arch_{arch_str}_{opt_name}_lr_{lr}"

except Exception:
    # Sicherer Fallback, falls die Keras-Version eine andere Struktur erzwingt
    param_str = "keras_nn"

# Dateiname zusammenbauen und ungültige Zeichen entfernen
filename = f"Auswertung_train_{model_name}_{param_str}_{timestamp}.csv"
filename = re.sub(r'[\\/*?:"<>| ]', "_", filename)

# Als CSV speichern (inklusive Index)
#results.to_csv(filename, index=True, sep=";", decimal=",")


# --- Konsolen-Ausgabe (unverändert bzw. erweitert) ---
print("Auswertung Trainingsdatensatz:")
print(f"  - MAE x: {results['abs_error_x'].mean():.4f}")
print(f"  - MAE y: {results['abs_error_y'].mean():.4f}")
print(f"  - MAE z: {results['abs_error_z'].mean():.4f}")
print(f"  - MSE x: {results['mse_x'].mean():.4f}")
print(f"  - MSE y: {results['mse_y'].mean():.4f}")
print(f"  - MSE z: {results['mse_z'].mean():.4f}")

print(f"\n[INFO] Datei erfolgreich gespeichert unter: {filename}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
Auswertung Trainingsdatensatz:
  - MAE x: 4.8392
  - MAE y: 4.9902
  - MAE z: 4.1179
  - MSE x: 47.1243
  - MSE y: 67.7591
  - MSE z: 27.4127

[INFO] Datei erfolgreich gespeichert unter: Auswertung_train_Sequential_Arch_512-256-128-64-3_Adam_lr_0.004999999888241291_20260617_101752.csv


In [30]:
# Zeigt die ersten 20 Zeilen elegant an
results.style.format(
    {
        "x": "{:.2f}",
        "x_pred": "{:.2f}",
        "abs_error_x": "{:.2f}",
        "y": "{:.2f}",
        "y_pred": "{:.2f}",
        "abs_error_y": "{:.2f}",
        "z": "{:.2f}",
        "z_pred": "{:.2f}",
        "abs_error_z": "{:.2f}",
    },
    decimal=",",  # Zeigt auch im Notebook Kommas statt Punkte!
).background_gradient(
    subset=["abs_error_x", "abs_error_y", "abs_error_z"], cmap="Reds"
)

,x,y,z,x_pred,y_pred,z_pred,abs_error_x,abs_error_y,abs_error_z,mse_x,mse_y,mse_z,source_file
161,"120,00","12,50","17,50","127,90","7,77","15,72","7,90","4,73","1,78","62,472602","22,394093","3,159086",Rechteck_Lang_L_U_4.csv
2,"50,00","50,00","50,00","50,08","49,79","48,97","0,08","0,21","1,03","0,006057","0,044540","1,067714",4eck_g_O_V_u_3.csv
121,"54,50","47,50","20,00","54,44","49,88","17,50","0,06","2,38","2,50","0,003534","5,642908","6,274763",6eck_L_U_6.csv
115,"47,50","32,50","5,75","36,13","25,46","6,61","11,37","7,04","0,86","129,355393","49,493584","0,743739",Durch_L_U_D_U_6.csv
70,"87,50","47,50","20,00","80,82","44,24","16,62","6,68","3,26","3,38","44,636777","10,601365","11,441020",Lumi_L_U_11.csv
136,"64,61","46,35","20,00","62,43","46,87","17,96","2,18","0,52","2,04","4,762616","0,266033","4,144897",Endboss_L_U_S_0_3.csv
47,"31,75","50,00","25,00","29,11","43,38","20,88","2,64","6,62","4,12","6,968368","43,795046","16,996826",Rechteck_L_L_7.csv
27,"53,33","91,00","30,00","51,91","94,11","24,79","1,42","3,11","5,21","2,002978","9,702451","27,121418",3eck_L_L_1.csv
145,"46,35","81,39","20,00","41,45","77,93","17,48","4,90","3,46","2,52","24,036430","11,974079","6,340534",Endboss_L_L_S_Y_3.csv
86,"47,50","20,00","87,50","47,73","20,51","74,90","0,23","0,51","12,60","0,052803","0,257633","158,766114",Lumi_H_L_U_1.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error

# Definition der 4 Drop-Varianten
drop_variants = {
    "No Drop": [],
    "Variant 1": [21, 43, 92, 159],
    "Variant 2": [92, 99, 31, 169, 150, 43],
    "Variant 3": [17, 21, 43, 53, 62, 92, 129, 150, 159, 166, 169],
    "Variant 4": [15, 17, 21, 30, 35, 43, 53, 62, 67, 72, 73, 75, 91, 92, 129, 139, 142, 150, 159, 166, 169]
}

# Definition der 5 Modell-Konfigurationen aus Zelle 8c0899a8
model_configs = [
    {"name": "Model 1", "scaler": "robust", "layers": [128, 64], "l2": 0.1, "dropout": 0.0, "lr": 0.05},
    {"name": "Model 2", "scaler": "robust", "layers": [512, 256, 128, 64], "l2": 0.5, "dropout": 0.0, "lr": 0.05},
    {"name": "Model 3", "scaler": "robust", "layers": [512, 256, 128, 64], "l2": 0.1, "dropout": 0.0, "lr": 0.01},
    {"name": "Model 4", "scaler": "robust", "layers": [256, 128, 64], "l2": 0.1, "dropout": 0.0, "lr": 0.01},
    {"name": "Model 5", "scaler": "minmax", "layers": [512, 256, 128, 64], "l2": 0.0, "dropout": 0.1, "lr": 0.005}
]

results_list = []
target_cols = ['x', 'y', 'z']

for drop_name, drop_indices in drop_variants.items():
    # Daten laden und vorbereiten
    df_full = pd.read_csv('training_data_ready.csv')
    # Nur existierende Indizes droppen
    indices_to_drop = [i for i in drop_indices if i in df_full.index]
    df_exp = df_full.drop(indices_to_drop, axis=0)
    
    x_cols = [col for col in df_exp.columns if col not in target_cols + ['source_file','Objektname','Objektkategorie','OAufnahme','Schwierigkeit','Gewicht']]
    
    X = df_exp[x_cols]
    y = df_exp[target_cols]
    
    # Train-Test-Split mit Validation Split bei Training (80/20 test -> 90/10 val in train)
    X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1, random_state=42)
    
    for config in model_configs:
        print(f"Starte {drop_name} mit {config['name']}...")
        
        # Scaler wählen
        if config["scaler"] == "robust":
            scaler = RobustScaler()
        else:
            scaler = MinMaxScaler()
            
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)
        
        # Modell aufbauen
        model = models.Sequential()
        model.add(layers.Input(shape=(X_train_scaled.shape[1],)))
        
        for units in config["layers"]:
            model.add(layers.Dense(units, activation='relu', kernel_regularizer=regularizers.l2(config["l2"])))
            if config["dropout"] > 0:
                model.add(layers.Dropout(config["dropout"]))
                
        model.add(layers.Dense(3))
        
        optimizer = tf.keras.optimizers.Adam(learning_rate=config["lr"])
        model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
        
        early_stopping = EarlyStopping(monitor='val_loss', patience=30130, restore_best_weights=True, verbose=0)
        
        model.fit(
            X_train_scaled, y_train,
            epochs=1000,
            batch_size=8,
            validation_data=(X_val_scaled, y_val),
            callbacks=[early_stopping],
            verbose=0
        )
        
        # Evaluation
        preds_train = model.predict(X_train_scaled, verbose=0)
        preds_val = model.predict(X_val_scaled, verbose=0)
        preds_test = model.predict(X_test_scaled, verbose=0)
        
        mae_train = mean_absolute_error(y_train, preds_train, multioutput='raw_values')
        mae_val = mean_absolute_error(y_val, preds_val, multioutput='raw_values')
        mae_test = mean_absolute_error(y_test, preds_test, multioutput='raw_values')
        
        results_list.append({
            "Drop_Variant": drop_name,
            "Model_Config": config["name"],
            "Train_MAE_x": mae_train[0], "Train_MAE_y": mae_train[1], "Train_MAE_z": mae_train[2],
            "Val_MAE_x": mae_val[0], "Val_MAE_y": mae_val[1], "Val_MAE_z": mae_val[2],
            "Test_MAE_x": mae_test[0], "Test_MAE_y": mae_test[1], "Test_MAE_z": mae_test[2]
        })

results_df = pd.DataFrame(results_list)
display(results_df)

Starte Variant 1 mit Model 1...
Starte Variant 1 mit Model 2...
Starte Variant 1 mit Model 3...
Starte Variant 1 mit Model 4...
Starte Variant 1 mit Model 5...
Starte Variant 2 mit Model 1...
Starte Variant 2 mit Model 2...
Starte Variant 2 mit Model 3...
Starte Variant 2 mit Model 4...
Starte Variant 2 mit Model 5...
Starte Variant 3 mit Model 1...
Starte Variant 3 mit Model 2...
Starte Variant 3 mit Model 3...
Starte Variant 3 mit Model 4...
Starte Variant 3 mit Model 5...
Starte Variant 4 mit Model 1...
Starte Variant 4 mit Model 2...
Starte Variant 4 mit Model 3...
Starte Variant 4 mit Model 4...
Starte Variant 4 mit Model 5...


,Drop_Variant,Model_Config,Train_MAE_x,Train_MAE_y,Train_MAE_z,Val_MAE_x,Val_MAE_y,Val_MAE_z,Test_MAE_x,Test_MAE_y,Test_MAE_z
0,Variant 1,Model 1,6.872856,5.064664,3.850614,11.253633,10.976868,6.647290,15.992828,17.790411,5.648474
1,Variant 1,Model 2,9.430860,10.574533,7.708986,12.669606,10.471148,8.333323,13.010736,13.799831,8.404533
2,Variant 1,Model 3,8.401100,7.005739,3.726586,10.653716,11.893679,5.657839,14.808399,19.510145,4.585637
3,Variant 1,Model 4,2.754220,2.432581,1.470721,12.398517,11.347514,4.908802,16.020546,19.290430,4.136741
4,Variant 1,Model 5,2.387145,2.250408,2.534258,11.789126,9.205834,5.357276,13.939017,15.459526,3.435226
5,Variant 2,Model 1,4.861551,5.413119,4.072461,8.941564,6.958927,5.652222,18.527706,15.882441,6.375751
6,Variant 2,Model 2,8.266910,9.758586,5.673196,10.188065,9.613363,6.196022,16.559505,18.860239,4.486477
7,Variant 2,Model 3,1.734056,2.540660,1.661490,9.005139,6.471330,5.433445,13.042131,15.887002,4.272820
8,Variant 2,Model 4,4.358511,2.707293,1.910377,9.665963,10.631804,3.540843,15.476354,14.491940,3.899290
9,Variant 2,Model 5,5.754449,6.042865,4.090389,9.500254,9.120817,4.303108,13.063194,14.106884,4.368253
